# 學生學習與作業版｜任務 11：資料切分與基準模型

**姓名：**歐國澤 **學號／組別：**C114118111 **完成日期：**2026-11-26

**學習目標：** 切分訓練、驗證、測試資料，建立多數類別基準並檢查資料分布。

本單元與教師版「任務 11」相同標題、資料、教學順序與範例；除了明確標示的作業與填空，其餘教師版教材內容保留。對照重點：程式填空對應教師版「11.2 多數類別基準」。

### 如何完成
1. 先閱讀觀念與執行資料準備；各單元可以獨立啟動，不需要依賴上一單元的變數。
2. **💻 P 作業**：在指定程式 cell 補上 `____`，可先用提供的 Vibe Coding 提示詞詢問 AI。補完再執行，並執行後面的驗證 cell。未填時出現 `NameError` 是預期現象，先完成填空再繼續。
3. **📝 Q 作業**：查詢 AI、比較教材內容，填寫緊接的文字作答區；請記錄實際提示詞、自己的解釋與查證方式。
4. **🎯 M 小任務**：在任務 4、8、12、16 將前面幾單元整合成可執行的成果。不要只交 AI 回覆，要交實際程式、輸出與自己的解釋。
5. 把 Notebook 存到自己的 Google 雲端硬碟，完成後下載／依教師指示繳交 `.ipynb`；小任務保留輸出、文字與 AI 使用紀錄。

**案例限制：**本資料是公式生成的 60 筆模擬紀錄，不能推論真實學生或因果。缺失與 15 小時值在計分後加入。預測時欄位可得性是假設；全部資料已在前半段探索，後續切出的測試集僅示範流程，不能視為完全獨立的新資料。

**完成檢核：**本單元有 1 項程式填空、2 項 AI 查詢作答；若是任務 4、8、12、16，另外有 1 項整合小任務。請保留自己的輸出、回答和驗證紀錄。


## 共用資料準備
以下在記憶體重建與既有教材相同的資料。執行後可以直接開始本單元；不會覆寫 CSV。


In [1]:
# 匯入套件：numpy 建立隨機資料，pandas 管理表格，pyplot 繪圖
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 固定種子，並維持呼叫順序，以重現原教材的 60 筆模擬資料
rng = np.random.default_rng(42)
n = 60
departments = ["資訊管理", "企業管理", "金融管理"]
# 每個字典鍵是一個欄位；對應的值是 60 名學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)
# 成績依設定公式與隨機誤差建立；關係來自模擬規則，不能視為因果發現
noise = rng.normal(0, 5, n)
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)
# 故意在計分後加入資料品質問題；無法從表格還原計分時原始值
df.loc[[5, 17], "sleep_hours"] = np.nan
df.loc[8, "screen_time_hours"] = 15.0
print("模擬資料形狀（列, 欄）：", df.shape)


模擬資料形狀（列, 欄）： (60, 7)


## 機器學習共用準備
依標籤分層切成 60% 訓練、20% 驗證、20% 測試；建立流程時只在訓練集 `fit`。固定切分使後續任務可比較；本資料已探索過，測試分數不是嚴格的獨立評估。


In [2]:
# 匯入資料切分、前處理、模型流程與評估工具
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

# 複製原始資料；將小考低於 75 分定義為教學用正類（1）
model_df = df.copy()
model_df["needs_support"] = (model_df["quiz_score"] < 75).astype(int)
numeric_columns = ["study_hours", "sleep_hours", "screen_time_hours", "attendance_rate"]
category_columns = ["department"]
feature_columns = numeric_columns + category_columns
# 明確選欄，避免把 quiz_score 或目標 needs_support 當成特徵
X = model_df[feature_columns].copy()
y = model_df["needs_support"].copy()
# 固定隨機種子；使用標籤分層，先保留 20% 測試集
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
# 剩下 80% 的四分之一為 20% 驗證集，最後得到 60/20/20
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)
# 數值缺失用訓練集中位數；類別缺失以訓練集眾數填補後做 One-Hot
numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
category_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))])
# ColumnTransformer 指定各欄位的處理方式；fit 時僅從訓練集學習
preprocess = ColumnTransformer([("num", numeric_pipe, numeric_columns),
                                ("cat", category_pipe, category_columns)])
print("訓練／驗證／測試筆數：", len(X_train), len(X_val), len(X_test))


訓練／驗證／測試筆數： 36 12 12


## 11.1 切分後先檢查
**觀念說明：**訓練集用來學習，驗證集用來選方案，測試集保留到選定後。分層依標籤維持類別比例，但不能增加資料量。若預測未來，應按實際時間順序考慮切分；本資料沒有時間欄位。


In [3]:
# 用索引檢查三組資料筆數、類別分布與互不重疊
for name, x_part, y_part in [("train", X_train, y_train), ("validation", X_val, y_val), ("test", X_test, y_test)]:
    print(name, "筆數", len(x_part), "類別", y_part.value_counts().sort_index().to_dict())
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_val) + len(X_test) == len(df)


train 筆數 36 類別 {0: 28, 1: 8}
validation 筆數 12 類別 {0: 9, 1: 3}
test 筆數 12 類別 {0: 9, 1: 3}


## 📝 學生作業 11-Q1｜AI 查詢與作答：三組資料用途
**任務：**訓練、驗證、測試資料分別做什麼？為何測試集不應反覆用來選模型？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請用考試類比解釋訓練、驗證與測試集，並說明多次選模對測試評估的影響。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 11-Q1 專用作答區：請雙擊此文字儲存格填寫
- AI 工具及實際提示詞：Gemini；使用題目提供的提示詞詢問訓練、驗證、測試集的用途，並用考試類比說明為何測試集不應反覆用來選模型。
- AI 回答重點（請自行整理）：可以用考試準備來類比三組資料：**訓練集**是平時的「課本與習題」，讓模型學習資料中的規律與計算填補中位數等參數；**驗證集**是考前的「模擬考」，用來比較不同模型或調整超參數、挑選最佳方案；**測試集**則是「正式大考」，必須封存到所有模型與參數都定案後才能拿出來評估一次。如果反覆拿測試集的分數來挑選或修改模型，就等於偷看了正式考題去背答案，會讓測試分數過度樂觀、失去評估未來新資料泛化能力的意義。
- 我的理解與具體例子：在本單元 `student-11-004` 中，60 筆資料被切分成訓練集 `36` 筆（60%）、驗證集 `12` 筆（20%）與測試集 `12` 筆（20%）。我們在 `11-P1` 建立基準模型時，只能用 `X_train, y_train`（36 筆）去 `fit`，再用 `X_val, y_val`（12 筆）評估驗證 Accuracy 與 Recall；而 `X_test, y_test`（12 筆）則必須保留到最終模型選定後才能使用。
- 查證來源／方式及日期（無法查證請註明）：對照本單元教材 11.1 節觀念說明與 `student-11-006` 輸出（`train 36`、`validation 12`、`test 12` 筆數及索引互不重疊斷言）核對，日期：2026-11-27。
- 我檢查或修正 AI 回答的地方：直接套用本資料集 60 筆切成 36 / 12 / 12 筆的實際分配，以及 `11-P1` 使用 `X_val` 驗證的流程作為例子。


## 11.2 多數類別基準
訓練集中哪個類別最多，就一律預測該類；再在驗證集評估。準確率可能看似不錯，但正類 Recall 可能很低。


## 💻 學生作業 11-P1｜Vibe Coding 程式填空
**對應教師版：**任務 11「11.2 多數類別基準」的示範程式。

**任務：**讓 baseline 僅以 X_train 與 y_train 學習，再由後續程式預測驗證資料。

**執行方式：**找到下一個程式儲存格中的 `____`，使用自己撰寫或 AI 建議的程式補齊，保留其他已提供的程式與註解。補完後先執行該 cell，再執行緊接的自動檢查；後續單元內容可能依賴此變數。

**可使用的 Vibe Coding 提示詞：**
```text
我是 Python 與資料分析初學者，正在 Google Colab 使用 pandas、numpy、matplotlib 或 scikit-learn。
我的任務是：讓 baseline 僅以 X_train 與 y_train 學習，再由後續程式預測驗證資料。
請依我貼上的上下文，只協助補上 ____，保留指定變數名稱與後續程式，不修改原始 df。
請解釋每一步的作用，提供繁體中文註解及可手動驗證的方法。
不要虛構執行結果；如需額外資訊，先指出缺少什麼。
```
完成後記錄實際使用的提示詞與自己改動的地方。


In [4]:
# DummyClassifier 的 most_frequent 只從訓練標籤找出多數類別
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
base_val_pred = baseline.predict(X_val)
print("驗證 Accuracy：", accuracy_score(y_val, base_val_pred))
print("驗證正類 Recall：", recall_score(y_val, base_val_pred, zero_division=0))
print("混淆矩陣（列實際、欄預測；順序 0,1）：\n", confusion_matrix(y_val, base_val_pred, labels=[0,1]))


驗證 Accuracy： 0.75
驗證正類 Recall： 0.0
混淆矩陣（列實際、欄預測；順序 0,1）：
 [[9 0]
 [3 0]]


### ✅ 作業 11-P1｜執行驗證
執行下方程式，並同時檢查上方實際輸出。自動檢查只涵蓋主要條件；如果圖表、說明或資料來源不符任務，仍需人工修正。


In [5]:
assert hasattr(baseline, "classes_")
assert len(base_val_pred) == len(y_val)
print("檢查通過：基準已訓練且預測筆數正確。")


檢查通過：基準已訓練且預測筆數正確。


### ✍️ 作業 11-P1｜我的完成紀錄
- 我實際使用的提示詞（若未使用 AI 請註明）：使用上方提供的 Vibe Coding 提示詞，詢問如何讓 `baseline` 僅以 `X_train` 與 `y_train` 學習。
- 我補上的程式在做什麼：輸入訓練特徵 `X_train` 與訓練標籤 `y_train`，呼叫 `baseline.fit(X_train, y_train)` 讓多數類別基準模型從訓練集找出出現次數最多的類別，供後續對驗證集 `X_val` 進行預測。
- 執行或驗證遇到的問題與修正：沒有報錯，補完後直接執行 `11-P1` 與驗證 cell 皆順利通過。
- 檢查結果及我如何確認：因為訓練集中 `0` 類為多數，模型一律預測 `0`，因此驗證正類 Recall 為 `0.0`（混淆矩陣第 2 欄預測為 `1` 的數量全為 0），驗證 cell 也印出「檢查通過：基準已訓練且預測筆數正確。」。


## 📝 學生作業 11-Q2｜AI 查詢與作答：時間與泛化
**任務：**如果目標是預測未來，為何常依時間切分？本案例為何做不到？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請比較時間切分與隨機切分，並說明沒有時間欄位時不可假稱已進行時間驗證。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 11-Q2 專用作答區：請雙擊此文字儲存格填寫
- AI 工具及實際提示詞：Gemini；使用題目提供的提示詞詢問為何預測未來常依時間切分、與隨機切分的差異，以及為何本案例做不到。
- AI 回答重點（請自行整理）：在真實世界部署預測模型時，我們永遠是用「過去已發生的資料」去預測「未來尚未發生的事件」。若使用隨機切分，模型在訓練時會偷看到未來時間點的分布與趨勢（例如未來的考題難度或整體平均變化），導致驗證分數虛高；改用「時間切分（例如前 8 週訓練、第 9 週驗證、第 10 週測試）」才能真正模擬過去推估未來的泛化能力。若資料集本身沒有時間戳記，就絕對不能宣稱已經做過時間切分驗證。
- 我的理解與具體例子：在本資料集 `df` 中，60 筆資料只有 `student_id`（1 到 60 號）與各科系、作息行為和分數，完全沒有紀錄「小考日期」或「第幾週」的時間欄位，所以我們在 `student-11-004` 只能用 `train_test_split(..., stratify=y)` 做分層隨機切分，並必須誠實揭露此切分僅示範機器學習流程，無法驗證跨時間預測未來的能力。
- 查證來源／方式及日期（無法查證請註明）：對照本單元教材 11.1 節、11.3 節說明與 `df.columns` 欄位清單（確認無任何日期／時間欄位）核對，日期：2026-11-27。
- 我檢查或修正 AI 回答的地方：直接對照本資料集 `df` 的 7 個欄位確認無時間戳記，以及 `student-11-004` 採用 `stratify=y` 分層切分的具體作法。


## 11.3 獨立資料與時間切分
整份資料已在任務 1～8 被查看，後續測試集只是流程教學示範，不是完全獨立的評估。部署前需新收集、未參與分析與設計的新樣本。

**思考：**每日銷售資料若要預測未來，為何較早訓練、較晚驗證通常比隨機打亂更貼近情境？重複客戶還要檢查跨資料集是否共用同一人的資訊。


## 延伸：分層比例與小樣本
比例接近不代表每組都有足夠樣本；實際列出類別數量，尤其是驗證及測試中的正類數。時間資料也不能為了分層而任意打亂未來。


In [6]:
# 三組的正類筆數及比例，協助解讀評估指標
for label, target in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(label, "總數", len(target), "正類", int(target.sum()), "正類比例", round(target.mean(),3))


train 總數 36 正類 8 正類比例 0.222
validation 總數 12 正類 3 正類比例 0.25
test 總數 12 正類 3 正類比例 0.25


## 本單元學習收束與 AI 使用紀錄

請學生先用自己的話解釋一個關鍵觀念，再指出一個執行結果及驗證方式。記錄：使用的提示詞、AI 回答中檢查或修改的地方、遇到的問題、修正後的證據與本資料限制。

**通用提示詞**：我是 Python 初學者，請逐步協助我完成上面的任務；每段程式加繁體中文註解，說明輸入、處理與輸出，提供兩項可實際執行的驗證，保留原始資料，不要虛構執行結果。

### ✍️ 本單元收束與 AI 使用紀錄作答
- **關鍵觀念解釋：**訓練機器學習模型前必須先將資料切分為訓練集（學習參數）、驗證集（挑選模型）與測試集（最終一次性評估），且前處理與基準模型都只能用訓練集 `fit`；此外，建立多數類別基準模型能幫助我們看清「高準確率背後可能隱藏正類 Recall 為 0」的盲點。
- **執行結果及驗證方式：**在 11.1 節透過索引集合交集檢查確認訓練（36 筆）、驗證（12 筆）、測試（12 筆）三組互不重疊且總和為 60 筆；在 `11-P1` 以 `baseline.fit(X_train, y_train)` 訓練多數類別基準並預測驗證集，看到正類 Recall 為 `0.0`，且驗證 cell 印出「檢查通過：基準已訓練且預測筆數正確。」。
- **使用的提示詞：**使用單元內提供的 `11-Q1`、`11-Q2` 查詢提示詞以及 `11-P1` 的 Vibe Coding 提示詞。
- **AI 回答中檢查或修改的地方：**將 AI 說明三組資料分工與時間切分原理的通用範例，改寫成直接對應本資料集 36 / 12 / 12 筆分層切分結果與無時間戳記欄位的具體說明。
- **遇到的問題與修正後的證據：**無報錯；補完 `11-P1` 後執行驗證 cell 確認 `baseline` 已具備 `classes_` 屬性且預測長度與 `y_val` 一致。
- **本資料限制：**本資料集僅有 60 筆模擬紀錄，切分後驗證集與測試集各只有 12 筆（正類樣本極少，指標波動大），且資料中沒有時間欄位無法進行時間切分，加上全體 60 筆已在任務 1～8 探索過，因此後續測試集僅供示範評估流程，不能視為完全獨立的新資料。
